# Merging RAS and NFL Draft Dataset

In [101]:
import pandas as pd
import string
import re
import numpy as np

In [102]:
df_draft = pd.read_csv("../data/nfl_draft_data.csv", delimiter=";")
df_ras = pd.read_csv("../data/RAS_1987_present.csv")

In [103]:
print(len(df_draft))
df_draft['' == df_draft['name']]['rec_yds']

14163


Series([], Name: rec_yds, dtype: float64)

In [104]:
print(len(df_ras))
df_ras.head(2)

9751


,Link,Name,POS,College,Year,RAS,Alltime RAS,Round,Draft Team
0,"<a class=""nt_btn "" style=""color: rgba(0, 0, 0,...",Justin Fargas,RB,Southern California,2003,10.0,10.0,3,Raiders
1,"<a class=""nt_btn "" style=""color: rgba(0, 0, 0,...",David Buehler,PK,Southern California,2009,10.0,10.0,5,Cowboys


### Looking at problem player names

In [105]:
print(df_ras[df_ras['Name'] == 'A.J. Brown'])
print("")
print(df_draft[df_draft['name'] == 'A.J. Brown'])

                                                   Link        Name POS  \
1945  <a class="nt_btn " style="color: rgba(0, 0, 0,...  A.J. Brown  WR   

          College  Year   RAS  Alltime RAS  Round Draft Team  
1945  Mississippi  2019  8.59         8.89      2     Titans  

       year        name      college pos  height  weight  hand_size  \
11887  2019  A.J. Brown  Mississippi  WR    72.5   226.0       9.75   

       arm_length  wonderlic  40_yard  ...  college_pass_int  \
11887       32.88        NaN     4.49  ...               NaN   

       college_pass_cmp_pct  college_rush_yds  college_rush_td  \
11887                   NaN               0.0              0.0   

       college_rec_yds  college_rec_td college_tackles college_sacks  \
11887           2984.0            19.0             NaN           NaN   

       college_ints college_fumbles  
11887           NaN             NaN  

[1 rows x 59 columns]


### Make Name, Team, Round merge compliant

In [106]:
print(df_ras.dtypes)
print("")
print(df_draft.dtypes)

Link               str
Name               str
POS                str
College            str
Year             int64
RAS            float64
Alltime RAS    float64
Round            int64
Draft Team         str
dtype: object

year                          int64
name                            str
college                         str
pos                             str
height                      float64
weight                      float64
hand_size                   float64
arm_length                  float64
wonderlic                   float64
40_yard                     float64
bench_press                 float64
vert_leap                   float64
broad_jump                  float64
shuttle                     float64
3_cone                      float64
60yd_shuttle                float64
simple_pos                      str
tag                             str
drafted                        bool
player_id                       str
draft_round                 float64
draft_pick            

## Start with RAS Dataset 
### [Name]
### We want to make name 
<li>all lowercase</li>
<li>trim whitespace</li>
<li>contain just letters, no punctuation</li>

In [107]:
regex_pattern = f"[{re.escape(string.punctuation) + string.whitespace}]"

In [108]:
df_ras['name_for_merge'] = df_ras['Name'].str.lower().str.strip().str.replace(regex_pattern, "", regex=True)

# print(df_ras[df_ras['name_for_merge'] == 'ajbrown'])
print(df_ras[df_ras['name_for_merge'] == 'pauloconnor'])
# df_ras.head()

                                                   Link           Name POS  \
7300  <a class="nt_btn " style="color: rgba(0, 0, 0,...  Paul O'Connor  OG   

     College  Year   RAS  Alltime RAS  Round Draft Team name_for_merge  
7300   Miami  1987  2.31         1.85      5     Giants    pauloconnor  


In [109]:
df_ras[(df_ras['Draft Team'] == 'titans') & (df_ras['Year'] < 1999)]


,Link,Name,POS,College,Year,RAS,Alltime RAS,Round,Draft Team,name_for_merge


### [Drafting Team]

In [110]:
df_ras['Draft Team'] = df_ras['Draft Team'].str.lower()

In [111]:
df_ras['Draft Team'].unique()

<StringArray>
[   'raiders',    'cowboys',      'colts',     'giants',       'jets',
 'commanders',      'bills', 'buccaneers',     'eagles',    'bengals',
   'seahawks',   'panthers',    'packers',    'jaguars',      '49ers',
      'lions',     'chiefs',    'vikings',     'texans',     'titans',
     'browns',   'dolphins',    'falcons',      'bears',     'saints',
   'chargers',  'cardinals',   'steelers',    'broncos',   'patriots',
     'ravens',       'rams']
Length: 32, dtype: str

In [112]:
df_ras.head()

,Link,Name,POS,College,Year,RAS,Alltime RAS,Round,Draft Team,name_for_merge
0,"<a class=""nt_btn "" style=""color: rgba(0, 0, 0,...",Justin Fargas,RB,Southern California,2003,10.0,10.0,3,raiders,justinfargas
1,"<a class=""nt_btn "" style=""color: rgba(0, 0, 0,...",David Buehler,PK,Southern California,2009,10.0,10.0,5,cowboys,davidbuehler
2,"<a class=""nt_btn "" style=""color: rgba(0, 0, 0,...",Anthony Richardson,QB,Florida,2023,10.0,10.0,1,colts,anthonyrichardson
3,"<a class=""nt_btn "" style=""color: rgba(0, 0, 0,...",Lorenzo Carter,LB,Georgia,2018,10.0,10.0,3,giants,lorenzocarter
4,"<a class=""nt_btn "" style=""color: rgba(0, 0, 0,...",Zack Kuntz,TE,Old Dominion,2023,10.0,10.0,7,jets,zackkuntz


## NFL Draft Dataset

### Formatting

<li>Name</li>
<li>Team</li>
<li>Round</li>


### Start with Name
### Apply same formatting as the RAS dataset

In [113]:
df_draft['name_for_merge'] = df_draft['name'].str.lower().str.strip().str.replace(regex_pattern, "", regex=True)

#print(df_ras[df_ras['name_for_merge'] == 'ajbrown'])
# print(df_draft[df_draft['name_for_merge'] == 'pauloconnor'])
df_draft.head()

,year,name,college,pos,height,weight,hand_size,arm_length,wonderlic,40_yard,...,college_pass_cmp_pct,college_rush_yds,college_rush_td,college_rec_yds,college_rec_td,college_tackles,college_sacks,college_ints,college_fumbles,name_for_merge
0,1987,Mike Adams,Arizona State,CB,69.8,198.0,8.50,30.50,NaN,4.42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mikeadams
1,1987,John Adickes,Baylor,C,74.8,266.0,10.25,30.00,NaN,4.97,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,johnadickes
2,1987,Tommy Agee,Auburn,FB,71.8,217.0,9.00,30.75,NaN,NaN,...,NaN,1733.0,10.0,321.0,3.0,NaN,NaN,NaN,NaN,tommyagee
3,1987,David Alexander,Tulsa (OK),C,75.0,279.0,10.50,32.75,NaN,5.13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,davidalexander
4,1987,Lyneal Alston,Southern Mississippi,WR,72.1,202.0,10.00,33.00,NaN,4.64,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lynealalston


### Team


In [114]:
df_draft['team'].unique()

<StringArray>
[                       nan,            'Chicago Bears',
         'Seattle Seahawks',      'Philadelphia Eagles',
      'St. Louis Cardinals',            'New York Jets',
     'Tampa Bay Buccaneers',     'New England Patriots',
       'New Orleans Saints',          'New York Giants',
            'Detroit Lions',           'Houston Oilers',
       'Indianapolis Colts',      'San Francisco 49ers',
       'Cincinnati Bengals',       'San Diego Chargers',
        'Minnesota Vikings',      'Los Angeles Raiders',
         'Los Angeles Rams',           'Miami Dolphins',
         'Cleveland Browns',            'Buffalo Bills',
      'Pittsburgh Steelers',        'Green Bay Packers',
      'Washington Redskins',          'Atlanta Falcons',
           'Dallas Cowboys',       'Kansas City Chiefs',
           'Denver Broncos',        'Phoenix Cardinals',
        'Arizona Cardinals',          'Oakland Raiders',
     'Jacksonville Jaguars',        'Carolina Panthers',
           'St. L

### Need to convert the old team names to their current forms first

#### Ex: Tennessee/Houston Oilers is the former team name of the Tenneessee Titans
#### Other names like St. Louis Rams and Los Angeles Rams we will just take the last name grouping of the team name




In [115]:
washington_team_names = ['Washington Redskins', 'Washington Football Team']
houston_team_names = ['Houston Oilers', 'Tennessee Oilers'] 

In [116]:
df_draft['team'] = np.where(df_draft['team'].isin(washington_team_names), 'commanders', df_draft['team'])
df_draft['team'] = np.where(df_draft['team'].isin(houston_team_names), 'titans', df_draft['team'])

In [117]:
df_draft[df_draft['team'] == 'commanders']

,year,name,college,pos,height,weight,hand_size,arm_length,wonderlic,40_yard,...,college_pass_cmp_pct,college_rush_yds,college_rush_td,college_rec_yds,college_rec_td,college_tackles,college_sacks,college_ints,college_fumbles,name_for_merge
49,1987,Brian Davis,Nebraska,CB,73.1,189.0,8.25,30.50,NaN,4.45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,briandavis
114,1987,Alfred Jenkins,Arizona,RB,75.3,218.0,10.25,32.00,NaN,4.74,...,NaN,270.0,9.0,0.0,0.0,NaN,NaN,NaN,NaN,alfredjenkins
211,1987,Ed Simmons,Eastern Washington,OT,76.1,292.0,9.00,34.25,NaN,5.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,edsimmons
220,1987,Timmy Smith,Texas Tech,RB,70.6,216.0,9.00,27.75,NaN,NaN,...,NaN,1313.0,8.0,401.0,1.0,NaN,NaN,NaN,NaN,timmysmith
393,1988,Harold Hicks,San Diego State,FS,71.6,199.0,9.50,NaN,NaN,4.49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,haroldhicks
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12864,2021,Camaron Cheeseman,Michigan,LS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,camaroncheeseman
12873,2021,Samuel Cosmi,Texas,OT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,samuelcosmi
12926,2021,Darrick Forrest,Cincinnati,SS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,200.0,0.0,6.0,0.0,darrickforrest
13152,2021,Benjamin St-Juste,Minnesota,CB,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,62.0,0.0,0.0,0.0,benjaminstjuste


### next lets change the team name to match the RAS dataset

In [118]:
nfl_draft_regex_pattern = r'.*\b(\w+)$'

df_draft['team'] = df_draft['team'].str.replace(nfl_draft_regex_pattern, r"\1", regex=True).str.lower().str.strip()


In [119]:
df_draft['team'].unique()

<StringArray>
[         nan,      'bears',   'seahawks',     'eagles',  'cardinals',
       'jets', 'buccaneers',   'patriots',     'saints',     'giants',
      'lions',     'titans',      'colts',      '49ers',    'bengals',
   'chargers',    'vikings',    'raiders',       'rams',   'dolphins',
     'browns',      'bills',   'steelers',    'packers', 'commanders',
    'falcons',    'cowboys',     'chiefs',    'broncos',    'jaguars',
   'panthers',     'ravens',     'texans']
Length: 33, dtype: str

### Lastly, we want to just convert the draft round column into an int to match with the RAS dataset

In [131]:
df_draft['draft_round'] = df_draft['draft_round'].astype('Int64')

print(df_draft['draft_round'].dtype)

print(df_ras['Round'].dtype)

Int64
int64
